# 🕵️‍♂️ Pakistani Social Media Content Detective
## Step 2: Clean the data

Run each cell in order. On Colab, Cell 1 will ask you to upload your two files. On your computer, it reads them from this folder (Step 2 already saved them here).

### Cell 1: Load your files (on Colab, select BOTH comments.csv and videos.csv)

In [1]:
import pandas as pd
import re

# On Colab, upload the files. On your computer, they are read from this folder.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()

comments = pd.read_csv("comments.csv")
videos = pd.read_csv("videos.csv")
print(f"Loaded {len(comments)} comments and {len(videos)} videos")

Loaded 8460 comments and 114 videos


### Cell 2: Remove junk comments
Write down the numbers this cell prints. They go straight into your report.

In [2]:
print("Starting comments:", len(comments))

# 1) Remove empty comments and extra spaces
comments = comments.dropna(subset=["comment"])
comments["comment"] = comments["comment"].astype(str).str.strip()
comments["comment"] = comments["comment"].str.replace(r"\s+", " ", regex=True)
print("After removing empty comments:", len(comments))

# 2) Remove spam comments that contain links
comments = comments[~comments["comment"].str.contains(r"http|www\.", case=False)]
print("After removing link/spam comments:", len(comments))

# 3) Remove emoji-only or very short comments (fewer than 3 letters)
def letter_count(text):
    return sum(ch.isalpha() for ch in text)

comments = comments[comments["comment"].apply(letter_count) >= 3]
print("After removing emoji-only / very short comments:", len(comments))

# 4) Remove duplicate comments (same text posted more than once)
comments["comment_lower"] = comments["comment"].str.lower()
comments = comments.drop_duplicates(subset="comment_lower").reset_index(drop=True)
print("After removing duplicates:", len(comments))
print(f"\nFINAL clean comments: {len(comments)}")

Starting comments: 8460
After removing empty comments: 8460
After removing link/spam comments: 8406
After removing emoji-only / very short comments: 8195
After removing duplicates: 7915

FINAL clean comments: 7915


### Cell 3: Detect the language of each comment
- **Urdu**: written in Urdu script
- **Roman Urdu**: Urdu written in English letters (e.g. "bhai price kya hai")
- **English**: everything else

In [3]:
ROMAN_URDU_WORDS = {
    "hai", "hain", "hy", "nahi", "nhi", "bhai", "bhi", "kya", "kia", "yeh", "ye",
    "acha", "achi", "acchi", "bohat", "bahut", "bht", "mein", "se", "ko", "ka",
    "ki", "ke", "aur", "gari", "gaari", "gadi", "wala", "wali", "sahi", "kar",
    "karo", "tha", "thi", "aap", "ap", "apna", "koi", "kab", "kitni", "kitna",
    "kitne", "lakh", "lakhs", "hoga", "hogi", "chahiye", "zabardast", "mehngi",
    "sasti", "pakistan", "yar", "yaar", "jee", "ji", "kuch", "sab", "wo", "woh",
}

def detect_language(text):
    if re.search(r"[\u0600-\u06FF]", text):
        return "Urdu"
    words = re.findall(r"[a-z]+", text.lower())
    if any(w in ROMAN_URDU_WORDS for w in words):
        return "Roman Urdu"
    return "English"

comments["language"] = comments["comment"].apply(detect_language)
print(comments["language"].value_counts())
comments[["comment", "language"]].sample(min(10, len(comments)))

language
English       4698
Roman Urdu    2956
Urdu           261
Name: count, dtype: int64


,comment,language
5841,"I have this car in Australia, already driven 1...",English
4133,Eeeee paint as,English
5455,This Car Name Should Be Porsh.alto as it creat...,English
6367,Had ha bhai smj nHe Ara apko hua kia ha Ai Q u...,Roman Urdu
5119,English bolne ki poori koshish par jab SACREEN...,Roman Urdu
4529,This is the cheap quality product. The price i...,English
3548,Exterior look ❤of this 🚗 car ❤,English
1994,frist,English
6393,I think good car in this price range ❤,English
3209,Why some stupids are saying 5.5M is good price...,English


### Cell 4: Add video info to each comment, then save

In [4]:
video_info = videos[["video_id", "title", "views", "likes"]].rename(
    columns={"views": "video_views", "likes": "video_likes"}
)
comments = comments.merge(video_info, on="video_id", how="left")
comments = comments.drop(columns="comment_lower")

comments.to_csv("comments_clean.csv", index=False, encoding="utf-8-sig")
if IN_COLAB:
    files.download("comments_clean.csv")
print("Saved comments_clean.csv! We'll use it in Step 4 (sentiment analysis).")
comments.head()

Saved comments_clean.csv! We'll use it in Step 4 (sentiment analysis).


,video_id,comment,comment_likes,replies,published,language,title,video_views,video_likes
0,ziOGUS2GnI0,"wo sub to thk hai , front number plate k ""E"" k...",0,0,2026-10-24T08:33:33Z,Roman Urdu,iCAUR V27 | Test Drive Review Pakwheels,65913,748
1,ziOGUS2GnI0,Cherry Tico9. VS Fortuner,0,0,2026-10-24T06:25:09Z,English,iCAUR V27 | Test Drive Review Pakwheels,65913,748
2,ziOGUS2GnI0,Defender Copy,0,0,2026-10-24T06:05:06Z,English,iCAUR V27 | Test Drive Review Pakwheels,65913,748
3,ziOGUS2GnI0,Nice,0,0,2026-10-24T04:54:47Z,English,iCAUR V27 | Test Drive Review Pakwheels,65913,748
4,ziOGUS2GnI0,"for the price, good entry ... LC200 now appear...",0,0,2026-10-24T03:54:35Z,English,iCAUR V27 | Test Drive Review Pakwheels,65913,748
